## 2. Install dependencies


In [ ]:
!pip install -q "hopsworks[python]==5.0.*" deltalake==0.22.3 prophet==1.1.6 cmdstanpy==1.3.0 \
    optuna==4.1.0 xgboost==2.1.1 lightgbm==4.5.0 shap==0.46.0 scikit-learn==1.5.2 \
    pandas==2.2.3 numpy==1.26.4 matplotlib==3.9.2 joblib
print("Install done.")


In [ ]:
!pip install -q --force-reinstall "numpy==1.26.4"

In [ ]:
!pip install --upgrade --force-reinstall "protobuf>=4.25.3,<5.0.0" "numpy<2.0.0" tensorflow hopsworks

## 3. Config + Hopsworks login


In [ ]:
import os
from google.colab import userdata
import hopsworks

HOPSWORKS_PROJECT_NAME = "api_my_project"
HOPSWORKS_HOST = "eu-west.cloud.hopsworks.ai"
FEATURE_GROUP_NAME = "aqi_features"
FEATURE_GROUP_VERSION = 4  # v4 = features on a strict hourly grid (see below)
MODEL_NAME = "aqi_forecaster"
TARGET_HORIZONS = (24, 48, 72)
N_TRIALS = 35
RANDOM_STATE = 42
SEQUENCE_LENGTH = 24

# OpenWeather's air pollution archive changes character on 2025-04-04: mean
# hourly |AQI change| collapses from ~46 to ~4.5 and never recovers, and the
# same break shows up in pm2_5. Everything before it is a different
# data-generating process, so training across the break teaches a volatility
# that no longer exists — that is what made every model lose to persistence.
# Set to None to deliberately train on all history.
TRAIN_START_DATE = "2025-04-04"

# Shrink predicted deltas toward 0 by a factor fitted on VALIDATION
# (0 = persistence, 1 = raw model). Predicted deltas are noisy, and an
# over-confident delta hurts the many hours where AQI barely moves — which is
# exactly where persistence is near-exact and where the models lost on MAE.
USE_DELTA_SHRINKAGE = True

HOPSWORKS_API_KEY = userdata.get("HOPSWORKS_API_KEY")
assert HOPSWORKS_API_KEY, "Add HOPSWORKS_API_KEY in Colab Secrets and enable Notebook access"

project = hopsworks.login(
    host=HOPSWORKS_HOST,
    api_key_value=HOPSWORKS_API_KEY,
    project=HOPSWORKS_PROJECT_NAME,
)
fs = project.get_feature_store()
mr = project.get_model_registry()
print("Logged in. Using FG", FEATURE_GROUP_NAME, "v" + str(FEATURE_GROUP_VERSION))


## 4. Load FG v4 + verify delta columns


In [ ]:
import pandas as pd
import numpy as np

fg = fs.get_feature_group(name=FEATURE_GROUP_NAME, version=FEATURE_GROUP_VERSION)
df = fg.read()
df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)
df = df.sort_values("timestamp").reset_index(drop=True)

needed = [f"aqi_delta_{h}h" for h in TARGET_HORIZONS] + [f"aqi_target_{h}h" for h in TARGET_HORIZONS]
missing = [c for c in needed if c not in df.columns]
assert not missing, f"FG v{FEATURE_GROUP_VERSION} missing columns {missing}. Re-run push-features locally."

# The hourly pipeline writes each hour as soon as its FEATURES are complete,
# which is up to 72h before its targets can exist, so the newest rows in the FG
# carry NULL targets by design — they are what inference reads. Here they'd be
# NaN labels: they land in the test split, and sklearn's metrics reject NaN, so
# the first persistence_baseline() call would fail before any model is scored.
before = len(df)
df = df.dropna(subset=needed).reset_index(drop=True)
if before != len(df):
    print(f"Dropped {before - len(df)} row(s) whose targets are still in the future")

print(df.shape)
print("Range:", df["timestamp"].min(), "->", df["timestamp"].max())
print("Delta cols OK:", needed)

# v4 is built on a strict hourly grid, so every remaining row's lags and targets
# really do span the hours they claim. Rows adjacent to long outages are dropped
# instead, which is why the count is lower than v3's.
step_hours = df["timestamp"].diff().dt.total_seconds().div(3600)
print(f"row gaps > 1h: {int((step_hours > 1).sum())} "
      f"(dropped outage neighbourhoods — expected on v4)")
df.head()


## 5. Split + feature columns


In [ ]:
def _snap_to_june_first(ts):
    year = ts.year if (ts.month, ts.day) >= (6, 1) else ts.year - 1
    return pd.Timestamp(year=year, month=6, day=1)

def chronological_split(df):
    df = df.sort_values("timestamp").reset_index(drop=True)
    max_date = df["timestamp"].max()
    test_start = _snap_to_june_first(max_date - pd.DateOffset(years=1))
    val_start = _snap_to_june_first(test_start - pd.DateOffset(years=1))
    train_df = df[df["timestamp"] < val_start]
    val_df = df[(df["timestamp"] >= val_start) & (df["timestamp"] < test_start)]
    test_df = df[df["timestamp"] >= test_start]
    if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
        # Post-regime-break history is ~16 months, too short for a June->June
        # split, so fall back to fractions. Note the test window then has no
        # smog season, which makes R2 harsh (low variance in the denominator).
        n = len(df)
        a, b = int(0.7 * n), int(0.85 * n)
        print("Window too short for a season-aligned split — using 70/15/15 fractions")
        return (df.iloc[:a].reset_index(drop=True),
                df.iloc[a:b].reset_index(drop=True),
                df.iloc[b:].reset_index(drop=True))
    print(f"train {train_df['timestamp'].min().date()}->{val_start.date()} | "
          f"val {val_start.date()}->{test_start.date()} | "
          f"test {test_start.date()}->{max_date.date()}")
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

def feature_columns(df):
    drop = {"timestamp", "hour", "month", "openweather_aqi_category"}
    for h in TARGET_HORIZONS:
        drop.add(f"aqi_target_{h}h")
        drop.add(f"aqi_delta_{h}h")
    return [c for c in df.columns if c not in drop]

if TRAIN_START_DATE:
    cutoff = pd.Timestamp(TRAIN_START_DATE)
    before = len(df)
    df = df[df["timestamp"] >= cutoff].reset_index(drop=True)
    print(f"TRAIN_START_DATE={cutoff.date()} -> kept {len(df)} of {before} rows "
          f"(test period is unaffected, only train shrinks)")

train_df, val_df, test_df = chronological_split(df)
feature_cols = feature_columns(df)
print("n_features:", len(feature_cols))
print("rows train/val/test:", len(train_df), len(val_df), len(test_df))
assert "aqi" in feature_cols, "aqi must stay a feature; it is also the reconstruction anchor"

# Train and test must come from the SAME data regime, or the models learn a
# volatility the test period doesn't have. These three numbers should be close.
for label, part in (("train", train_df), ("val", val_df), ("test", test_df)):
    d1 = part["aqi"].diff().abs()
    print(f"  {label:5} {part['timestamp'].min().date()} -> {part['timestamp'].max().date()} "
          f"| aqi mean={part['aqi'].mean():6.1f} | mean|d1h|={d1.mean():5.2f}")


## 6. Evaluation helpers (absolute AQI metrics)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import os
os.makedirs("reports", exist_ok=True)

def evaluate(y_true, y_pred, name):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "model": name,
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }

def reconstruct_absolute(aqi_now, y_delta_pred, shrinkage=1.0):
    """absolute AQI = current AQI + shrinkage * predicted delta.

    shrinkage=1 is the raw model, shrinkage=0 collapses exactly to persistence.
    aqi_now must be the UNSCALED current AQI: `aqi` is itself a model feature,
    so scaling a frame in place standardizes it, and anchoring on that adds ~0
    instead of the real AQI level, silently reducing the prediction to the
    bare delta.
    """
    return (np.asarray(aqi_now, dtype=float)
            + float(shrinkage) * np.asarray(y_delta_pred, dtype=float))

SHRINKAGE_GRID = tuple(float(x) for x in np.round(np.linspace(0.0, 1.0, 21), 2))

def fit_delta_shrinkage(aqi_now, y_absolute_true, y_delta_pred, grid=SHRINKAGE_GRID):
    """Pick the delta multiplier on VALIDATION data — never on test.

    Scored on relative RMSE + relative MAE against the lambda=0 point, because
    the registry gate needs RMSE, MAE and R2 all better than persistence at
    once (R2 is monotone in RMSE for a fixed y_true, so it needs no own term).
    """
    eps = 1e-12
    base = evaluate(y_absolute_true, reconstruct_absolute(aqi_now, y_delta_pred, 0.0), "lam=0")
    best_lam, best_score, best_m = 0.0, 1.0, base
    for lam in grid:
        m = evaluate(y_absolute_true, reconstruct_absolute(aqi_now, y_delta_pred, lam), f"lam={lam}")
        score = 0.5 * (m["RMSE"] / max(base["RMSE"], eps) + m["MAE"] / max(base["MAE"], eps))
        if score < best_score - eps:
            best_lam, best_score, best_m = float(lam), score, m
    return best_lam, {"shrinkage": best_lam, "val_rmse": best_m["RMSE"],
                      "val_mae": best_m["MAE"], "val_r2": best_m["R2"],
                      "val_persistence_rmse": base["RMSE"],
                      "val_persistence_mae": base["MAE"]}

def persistence_baseline(test_df, absolute_target_col):
    return evaluate(test_df[absolute_target_col], test_df["aqi"], "Persistence Baseline")

def beats_persistence(m, b):
    return m["RMSE"] < b["RMSE"] and m["MAE"] < b["MAE"] and m["R2"] > b["R2"]

def pick_best_candidate(results, baseline):
    eligible = [r for r in results if r["model"] != "Persistence Baseline" and beats_persistence(r, baseline)]
    if not eligible:
        return None
    eligible.sort(key=lambda r: (r["RMSE"], r["MAE"], -r["R2"]))
    return eligible[0]

print("helpers ready")


## 7. Model trainers (Optuna tabular + Prophet + LSTM/GRU)


In [ ]:
import optuna
import numpy as np
import pandas as pd
import logging
import os
import tensorflow as tf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import lightgbm as lgb
from prophet import Prophet
from tensorflow import keras

# Suppress logs
logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# -------------------------------------------------------------------
# 1. GPU & HARDWARE ACCELERATION SETUP
# -------------------------------------------------------------------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"TensorFlow GPU Detected: {gpus[0].name}")
    try:
        # Enable dynamic memory growth to prevent OOM
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print(f"GPU memory growth error: {e}")
else:
    print("TensorFlow GPU NOT detected. Running TF on CPU.")

# Check if CUDA is available for XGBoost
HAS_XGB_GPU = False
try:
    # Test small XGBoost model on CUDA
    xgb.XGBRegressor(tree_method="hist", device="cuda").fit(np.ones((10, 2)), np.ones(10))
    HAS_XGB_GPU = True
    print("XGBoost CUDA Acceleration Enabled.")
except Exception:
    print("XGBoost running on CPU (Multi-threaded).")

TABULAR = {"Ridge", "RandomForest", "XGBoost", "LightGBM"}
TREES = {"RandomForest", "XGBoost", "LightGBM"}

# -------------------------------------------------------------------
# 2. FAST CROSS-VALIDATION EVALUATOR
# -------------------------------------------------------------------
def _cv_rmse(model, X, y, tscv):
    scores = []
    # Convert to numpy for faster indexing inside the CV loop
    X_val, y_val = X.values, y.values
    for tr, va in tscv.split(X_val):
        model.fit(X.iloc[tr], y.iloc[tr])
        preds = model.predict(X.iloc[va])
        scores.append(np.sqrt(mean_squared_error(y_val[va], preds)))
    return float(np.mean(scores))

# -------------------------------------------------------------------
# 3. OPTIMIZED TABULAR TUNERS
# -------------------------------------------------------------------
def tune_ridge(X, y, tscv, n_trials=N_TRIALS):
    def obj(trial):
        pipe = make_pipeline(
            StandardScaler(),
            Ridge(alpha=trial.suggest_float("alpha", 0.01, 100, log=True), random_state=RANDOM_STATE)
        )
        return _cv_rmse(pipe, X, y, tscv)

    study = optuna.create_study(direction="minimize")
    study.optimize(obj, n_trials=n_trials, show_progress_bar=False)

    return make_pipeline(StandardScaler(), Ridge(**study.best_params, random_state=RANDOM_STATE)).fit(X, y)

def tune_rf(X, y, tscv, n_trials=N_TRIALS):
    def obj(trial):
        p = dict(
            n_estimators=trial.suggest_int("n_estimators", 100, 250, step=50),
            max_depth=trial.suggest_categorical("max_depth", [8, 12, 16]),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 8),
            max_samples=0.8,  # Subsampling speeds up Random Forest fitting drastically
            n_jobs=-1,         # Use all CPU cores in parallel
            random_state=RANDOM_STATE
        )
        return _cv_rmse(RandomForestRegressor(**p), X, y, tscv)

    study = optuna.create_study(direction="minimize")
    study.optimize(obj, n_trials=n_trials, show_progress_bar=False)

    return RandomForestRegressor(
        random_state=RANDOM_STATE,
        max_samples=0.8,
        n_jobs=-1,
        **study.best_params
    ).fit(X, y)

def tune_xgb(X, y, tscv, n_trials=N_TRIALS, objective="reg:squarederror"):
    """objective="reg:absoluteerror" optimizes MAE instead of squared error.

    Worth having as a SEPARATE candidate rather than a replacement: the L1 fit
    usually wins on MAE and gives up a little RMSE, and the registry gate wants
    both, so let the gate pick instead of deciding here.
    """
    device_param = "cuda" if HAS_XGB_GPU else "cpu"

    def obj(trial):
        p = dict(
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 3, 9),
            n_estimators=trial.suggest_int("n_estimators", 100, 400, step=50),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            objective=objective,
            tree_method="hist",
            device=device_param,
            n_jobs=-1 if device_param == "cpu" else 1,
            random_state=RANDOM_STATE
        )
        return _cv_rmse(xgb.XGBRegressor(**p), X, y, tscv)

    study = optuna.create_study(direction="minimize")
    study.optimize(obj, n_trials=n_trials, show_progress_bar=False)

    best_p = {
        **study.best_params,
        "objective": objective,
        "tree_method": "hist",
        "device": device_param,
        "random_state": RANDOM_STATE
    }
    return xgb.XGBRegressor(**best_p).fit(X, y)

def tune_lgb(X, y, tscv, n_trials=N_TRIALS, objective="regression"):
    """objective="regression_l1" optimizes MAE instead of squared error.

    subsample_freq=1 is required for subsample to do anything at all — LightGBM
    defaults it to 0, which disables bagging and made the tuned `subsample`
    silently inert.
    """
    def obj(trial):
        p = dict(
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int("num_leaves", 16, 96),
            n_estimators=trial.suggest_int("n_estimators", 100, 400, step=50),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            subsample_freq=1,
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            objective=objective,
            n_jobs=-1,
            verbosity=-1,
            random_state=RANDOM_STATE
        )
        return _cv_rmse(lgb.LGBMRegressor(**p), X, y, tscv)

    study = optuna.create_study(direction="minimize")
    study.optimize(obj, n_trials=n_trials, show_progress_bar=False)

    return lgb.LGBMRegressor(
        random_state=RANDOM_STATE,
        subsample_freq=1,
        objective=objective,
        n_jobs=-1,
        verbosity=-1,
        **study.best_params
    ).fit(X, y)

# -------------------------------------------------------------------
# 4. PROPHET & RNN TRAINERS
# -------------------------------------------------------------------
def train_prophet(fit_df, test_df, abs_col, horizon):
    """Prophet stays on ABSOLUTE AQI by design — it decomposes a level series
    into trend + seasonality, and a mean-reverting delta has no trend to
    decompose.

    fit_df must run up to the start of the test period (pass train + val).
    Fitting on train alone and predicting test timestamps a year or more later
    makes Prophet extrapolate its linear trend far past any data, which on
    Lahore's steeply declining AQI produced RMSE ~2100 on a 0-1000 scale.
    """
    pt = (fit_df[["timestamp", "aqi"]]
          .rename(columns={"timestamp": "ds", "aqi": "y"})
          .dropna().drop_duplicates(subset="ds").sort_values("ds").reset_index(drop=True))

    lead_days = ((test_df["timestamp"].min() + pd.Timedelta(hours=horizon))
                 - pt["ds"].max()).total_seconds() / 86400
    if lead_days > 7:
        print(f"    WARNING: Prophet's first forecast point is {lead_days:.0f} days past "
              f"its fit end — it is extrapolating, results are not comparable.")

    model = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
    model.fit(pt)
    future = pd.DataFrame({"ds": test_df["timestamp"] + pd.Timedelta(hours=horizon)})
    y_pred = model.predict(future)["yhat"].values
    return model, evaluate(test_df[abs_col], y_pred, "Prophet")

def make_sequences(df, feature_cols, target_col, seq_len=SEQUENCE_LENGTH,
                   anchor=None, absolute_target=None):
    """Window i covers rows [i-seq_len+1 .. i] INCLUSIVE and predicts row i.

    Ending the window ON the prediction origin matters for delta targets: the
    delta is defined relative to the AQI at row i, so a window stopping at i-1
    asks the net for a delta against a value it never saw, while the tabular
    models do get row i.

    `anchor` / `absolute_target` are optional row-aligned arrays that get sliced
    identically. They must be captured BEFORE feature scaling (see
    reconstruct_absolute) — this function deliberately never reads them off the
    dataframe, so the scaled-anchor bug can't come back.
    """
    vals = np.asarray(df[feature_cols].values, dtype=float)
    tgts = np.asarray(df[target_col].values, dtype=float)
    n = len(df)
    end_idx = np.arange(seq_len - 1, n)

    windows = np.lib.stride_tricks.sliding_window_view(vals, window_shape=seq_len, axis=0)
    X = np.ascontiguousarray(windows.transpose(0, 2, 1))  # (samples, timesteps, features)
    y = tgts[end_idx]

    anchor_out = None if anchor is None else np.asarray(anchor, dtype=float)[end_idx]
    abs_out = (None if absolute_target is None
               else np.asarray(absolute_target, dtype=float)[end_idx])
    return X, y, anchor_out, abs_out

def _build_rnn(kind, n_feat):
    Layer = keras.layers.LSTM if kind == "LSTM" else keras.layers.GRU
    model = keras.Sequential([
        keras.layers.Input(shape=(SEQUENCE_LENGTH, n_feat)),
        Layer(64, return_sequences=True),
        keras.layers.Dropout(0.2),
        Layer(32),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    return model

def train_rnn(kind, train_df, val_df, test_df, feature_cols, delta_col, abs_col):
    """Train one recurrent net on the delta target, score on absolute AQI.

    The delta target is standardized before the MSE loss sees it (the features
    already are), so gradient scales match and the net doesn't burn its early
    epochs just learning the output magnitude.

    Returns (model, metrics, meta) — meta carries the val/test delta predictions
    and their unscaled anchors so a shrinkage factor can be fitted on val.
    """
    # Anchors and absolute truths are read BEFORE scaling touches the frames.
    anchors = {k: d["aqi"].to_numpy(dtype=float)
               for k, d in (("val", val_df), ("test", test_df))}
    abs_true = {k: d[abs_col].to_numpy(dtype=float)
                for k, d in (("val", val_df), ("test", test_df))}

    scaler = StandardScaler().fit(train_df[feature_cols])
    tr, va, te = train_df.copy(), val_df.copy(), test_df.copy()
    for frame in (tr, va, te):
        frame[feature_cols] = scaler.transform(frame[feature_cols])

    Xtr, ytr, _, _ = make_sequences(tr, feature_cols, delta_col)
    Xva, yva, va_anchor, va_abs = make_sequences(
        va, feature_cols, delta_col, anchor=anchors["val"], absolute_target=abs_true["val"])
    Xte, yte, te_anchor, te_abs = make_sequences(
        te, feature_cols, delta_col, anchor=anchors["test"], absolute_target=abs_true["test"])

    target_scaler = StandardScaler().fit(ytr.reshape(-1, 1))
    ytr_s = target_scaler.transform(ytr.reshape(-1, 1)).ravel()
    yva_s = target_scaler.transform(yva.reshape(-1, 1)).ravel()

    model = _build_rnn(kind, len(feature_cols))
    model.fit(
        Xtr, ytr_s,
        validation_data=(Xva, yva_s),
        epochs=100,
        batch_size=128,  # larger batch for GPU throughput
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                                 restore_best_weights=True)],
        verbose=0
    )

    def _delta(X):
        raw = model.predict(X, batch_size=256, verbose=0).reshape(-1, 1)
        return target_scaler.inverse_transform(raw).ravel()

    val_delta, test_delta = _delta(Xva), _delta(Xte)
    metrics = evaluate(te_abs, reconstruct_absolute(te_anchor, test_delta), kind)
    meta = {
        "scaler": scaler, "target_scaler": target_scaler,
        "sequence_length": SEQUENCE_LENGTH,
        "val_delta_pred": val_delta, "val_anchor": va_anchor, "val_absolute_true": va_abs,
        "test_delta_pred": test_delta, "test_anchor": te_anchor, "test_absolute_true": te_abs,
    }
    return model, metrics, meta

print("Optimized trainers ready.")

## 8. Train all horizons (24 / 48 / 72) + register winners


In [ ]:
import os
import shutil
import cmdstanpy
import prophet

# 1. Install/Verify CmdStan via cmdstanpy
cmdstanpy.install_cmdstan()

# 2. Get the working CmdStan installation path
working_cmdstan_path = cmdstanpy.cmdstan_path()

# 3. Get Prophet's expected broken path
prophet_dir = os.path.dirname(prophet.__file__)
expected_cmdstan_dir = os.path.join(prophet_dir, "stan_model", "cmdstan-2.33.1")

# 4. Remove the broken bundled directory and link it to the working build
if os.path.exists(expected_cmdstan_dir) or os.path.islink(expected_cmdstan_dir):
    if os.path.islink(expected_cmdstan_dir):
        os.unlink(expected_cmdstan_dir)
    else:
        shutil.rmtree(expected_cmdstan_dir)

os.symlink(working_cmdstan_path, expected_cmdstan_dir)
print("Successfully linked CmdStan to Prophet!")

In [ ]:
import joblib
import shap

SHRUNK_SUFFIX = "_shrunk"
L1_SUFFIX = "_L1"
SERVEABLE_KINDS = {"tabular", "ensemble"}  # /predict feeds one flat feature row


def base_model_name(name):
    """Strip variant suffixes so XGBoost, XGBoost_L1 and XGBoost_shrunk count
    as one model — otherwise an "ensemble" can be three views of one fit, and
    the SHAP/TREES check misses the L1 variants."""
    for suffix in (SHRUNK_SUFFIX, L1_SUFFIX):
        if name.endswith(suffix):
            name = name[:-len(suffix)]
    return name

def add_delta_candidate(candidates, results, *, name, kind, model, h,
                        val_delta_pred, val_anchor, val_absolute_true,
                        test_delta_pred, test_anchor, test_absolute_true):
    """Score a delta model on absolute AQI, then add its val-shrunk twin.

    Both variants share the same fitted model — only the multiplier on the
    predicted delta differs — so the extra candidate costs no extra training.
    """
    raw_pred = reconstruct_absolute(test_anchor, test_delta_pred)
    m = evaluate(test_absolute_true, raw_pred, name)
    m["horizon_hours"], m["shrinkage"] = h, 1.0
    results.append(m)
    candidates[name] = {"model": model, "kind": kind, "shrinkage": 1.0,
                        "abs_pred": raw_pred, "metrics": m}
    print(f"    {name}: RMSE={m['RMSE']:.2f} MAE={m['MAE']:.2f} R2={m['R2']:.3f}")

    if not USE_DELTA_SHRINKAGE:
        return
    lam, diag = fit_delta_shrinkage(val_anchor, val_absolute_true, val_delta_pred)
    print(f"      val-fitted shrinkage lambda={lam:.2f} (val RMSE {diag['val_rmse']:.2f} "
          f"vs val persistence {diag['val_persistence_rmse']:.2f})")
    # lambda=1 duplicates the raw row; lambda=0 IS persistence and can never
    # pass a strictly-better gate.
    if lam <= 0.0 or lam >= 1.0:
        return
    sname = name + SHRUNK_SUFFIX
    spred = reconstruct_absolute(test_anchor, test_delta_pred, lam)
    sm = evaluate(test_absolute_true, spred, sname)
    sm["horizon_hours"], sm["shrinkage"] = h, lam
    results.append(sm)
    candidates[sname] = {"model": model, "kind": kind, "shrinkage": lam,
                         "abs_pred": spred, "metrics": sm}
    print(f"    {sname}: RMSE={sm['RMSE']:.2f} MAE={sm['MAE']:.2f} R2={sm['R2']:.3f}")

tscv = TimeSeriesSplit(n_splits=5)
all_rows = []
registry_summary = []

for h in TARGET_HORIZONS:
    print("\n" + "=" * 70)
    print(f"HORIZON {h}h  (day {h // 24})")
    print("=" * 70)
    delta_col = f"aqi_delta_{h}h"
    abs_col = f"aqi_target_{h}h"

    baseline = persistence_baseline(test_df, abs_col)
    baseline["horizon_hours"] = h
    baseline["shrinkage"] = 0.0  # persistence IS the delta=0 prediction
    results = [baseline]
    candidates = {}

    X_train, y_delta = train_df[feature_cols], train_df[delta_col]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]
    y_abs = test_df[abs_col].to_numpy(dtype=float)
    aqi_test = test_df["aqi"].to_numpy(dtype=float)
    aqi_val = val_df["aqi"].to_numpy(dtype=float)
    y_val_abs = val_df[abs_col].to_numpy(dtype=float)

    # Prophet on absolute levels, fitted through the END OF VAL so it only ever
    # forecasts `h` hours past its own data instead of extrapolating for years.
    print("  fitting Prophet...")
    _, prophet_m = train_prophet(pd.concat([train_df, val_df], ignore_index=True),
                                 test_df, abs_col, h)
    prophet_m["horizon_hours"] = h
    results.append(prophet_m)
    print(f"    Prophet: RMSE={prophet_m['RMSE']:.2f} MAE={prophet_m['MAE']:.2f} R2={prophet_m['R2']:.3f}")

    # Tabular on deltas. The _L1 pair fits the same models under absolute-error
    # loss, which targets MAE — the metric persistence beats us on hardest.
    for name, tuner in [("Ridge", tune_ridge), ("RandomForest", tune_rf),
                        ("XGBoost", tune_xgb), ("LightGBM", tune_lgb),
                        ("XGBoost_L1",
                         lambda X, y, cv: tune_xgb(X, y, cv, objective="reg:absoluteerror")),
                        ("LightGBM_L1",
                         lambda X, y, cv: tune_lgb(X, y, cv, objective="regression_l1"))]:
        print(f"  tuning {name}...")
        model = tuner(X_train, y_delta, tscv)
        add_delta_candidate(
            candidates, results, name=name, kind="tabular", model=model, h=h,
            val_delta_pred=model.predict(X_val), val_anchor=aqi_val,
            val_absolute_true=y_val_abs,
            test_delta_pred=model.predict(X_test), test_anchor=aqi_test,
            test_absolute_true=y_abs,
        )

    # LSTM / GRU on deltas. Their windowed predictions are SEQUENCE_LENGTH-1
    # rows shorter than the tabular ones, so they stay out of the ensembles.
    for kind in ("LSTM", "GRU"):
        print(f"  training {kind}...")
        model, _, meta = train_rnn(kind, train_df, val_df, test_df,
                                   feature_cols, delta_col, abs_col)
        add_delta_candidate(
            candidates, results, name=kind, kind="recurrent",
            model=(model, meta), h=h,
            val_delta_pred=meta["val_delta_pred"], val_anchor=meta["val_anchor"],
            val_absolute_true=meta["val_absolute_true"],
            test_delta_pred=meta["test_delta_pred"], test_anchor=meta["test_anchor"],
            test_absolute_true=meta["test_absolute_true"],
        )

    # Ensembles over the tabular candidates (raw and shrunk both eligible),
    # capped at one variant per base model so it can't be 3 copies of XGBoost.
    ranked = [n for n, c in sorted(candidates.items(), key=lambda kv: kv[1]["metrics"]["RMSE"])
              if c["kind"] == "tabular"]
    seen, members_top = set(), []
    for n in ranked:
        base = base_model_name(n)
        if base in seen:
            continue
        seen.add(base)
        members_top.append(n)
        if len(members_top) == 3:
            break

    seen_gb, xgb_lgbm = set(), []
    for n in ranked:
        base = base_model_name(n)
        if base not in {"XGBoost", "LightGBM"} or base in seen_gb:
            continue
        seen_gb.add(base)
        xgb_lgbm.append(n)

    for ens_name, members in [(f"Ensemble_top{len(members_top)}_tabular", members_top),
                              ("Ensemble_XGB_LGBM", xgb_lgbm)]:
        if len(members) < 2 or ens_name in candidates:
            continue
        ens = np.mean(np.vstack([candidates[m]["abs_pred"] for m in members]), axis=0)
        m = evaluate(y_abs, ens, ens_name)
        m["horizon_hours"] = h
        m["shrinkage"] = float(np.mean([candidates[m2]["shrinkage"] for m2 in members]))
        results.append(m)
        candidates[ens_name] = {
            "model": {"type": "mean_ensemble",
                      "members": {m2: {"model": candidates[m2]["model"],
                                       "shrinkage": candidates[m2]["shrinkage"]}
                                  for m2 in members}},
            "kind": "ensemble", "shrinkage": 1.0,  # members carry their own factors
            "abs_pred": ens, "metrics": m,
        }
        print(f"    {ens_name} ({', '.join(members)}): RMSE={m['RMSE']:.2f} "
              f"MAE={m['MAE']:.2f} R2={m['R2']:.3f}")

    rdf = pd.DataFrame(results).sort_values("RMSE")
    print(rdf.to_string(index=False))
    rdf.to_csv(f"reports/model_comparison_{h}h.csv", index=False)
    all_rows.append(rdf)

    winner = pick_best_candidate(results, baseline)
    if winner is None:
        print(f"NO model beat persistence on all 3 metrics for {h}h — nothing registered.")
        registry_summary.append({"horizon": h, "registered": False, "winner": None})
        continue

    # Prophet needs a `ds` series and the recurrent nets need a 24-hour window,
    # so neither can be served through /predict. If one wins overall, fall back
    # to the best serveable candidate that still beats persistence.
    reg_name = winner["model"]
    cand = candidates.get(reg_name)
    if cand is None or cand["kind"] not in SERVEABLE_KINDS:
        serveable_rows = [r for r in results
                          if candidates.get(r["model"], {}).get("kind") in SERVEABLE_KINDS]
        fallback = pick_best_candidate(serveable_rows, baseline)
        if fallback is None:
            print(f"Winner {reg_name} is not serveable and no serveable model beat "
                  f"persistence for {h}h — nothing registered.")
            registry_summary.append({"horizon": h, "registered": False, "winner": reg_name})
            continue
        print(f"Winner {reg_name} is not serveable through /predict; "
              f"registering {fallback['model']} instead.")
        winner, reg_name = fallback, fallback["model"]
        cand = candidates[reg_name]

    artifact_model = cand["model"]
    shrinkage = float(cand["shrinkage"])
    base_name = base_model_name(reg_name)

    if base_name in TREES:
        Xs = X_test.sample(n=min(2000, len(X_test)), random_state=RANDOM_STATE)
        shap.summary_plot(shap.TreeExplainer(artifact_model).shap_values(Xs), Xs, show=False)
        plt.savefig(f"reports/shap_summary_{h}h.png", bbox_inches="tight"); plt.close()

    registry_name = f"{MODEL_NAME}_{h}h"
    os.makedirs("model_artifact", exist_ok=True)
    payload = {
        "model": artifact_model,
        "feature_cols": feature_cols,
        "horizon_hours": h,
        "target_type": "delta",
        "model_name": reg_name,
        # Serving must apply the same factor training scored with, or the served
        # prediction won't be the model that passed the gate.
        "shrinkage": shrinkage,
    }
    path = f"model_artifact/{registry_name}.joblib"
    joblib.dump(payload, path)

    hw_model = mr.python.create_model(
        name=registry_name,
        metrics={"RMSE": winner["RMSE"], "MAE": winner["MAE"], "R2": winner["R2"]},
        description=(f"AQI {h}h day-{h//24} — {reg_name} "
                     f"(serve as aqi + {shrinkage:.2f} * predicted_delta)"),
    )
    hw_model.save("model_artifact")
    print(f"REGISTERED {registry_name} <- {reg_name} | RMSE={winner['RMSE']:.2f} "
          f"MAE={winner['MAE']:.2f} R2={winner['R2']:.3f} | shrinkage={shrinkage:.2f}")
    registry_summary.append({"horizon": h, "registered": True, "winner": reg_name,
                             "shrinkage": shrinkage,
                             **{k: winner[k] for k in ("RMSE", "MAE", "R2")}})

combined = pd.concat(all_rows, ignore_index=True)
combined.to_csv("reports/model_comparison.csv", index=False)
print("\n=== REGISTRY SUMMARY ===")
print(pd.DataFrame(registry_summary).to_string(index=False))
print("\n=== FULL COMPARISON ===")
print(combined.to_string(index=False))


## 9. Walk-forward robustness check (rolling origin)

Section 8 tests on **one** window, and after the regime filter that window is
monsoon: flat, low-variance, no smog season. A model that wins there might only
suit calm weather — and the reverse is just as possible.

This cuts the post-break timeline into successive blocks and tests each one
using only the data before it, so **smog season gets evaluated as a test period
without any future rows leaking into training**. Hyperparameters are fixed
rather than re-tuned per fold, so folds stay comparable and cheap; this measures
whether a win *holds across periods*, not peak accuracy.

Read `folds_won` first: a model that beats persistence in 1 of 4 folds is noise,
one that beats it in 4 of 4 is real.


In [ ]:
WF_FOLDS = 4
WF_TEST_DAYS = 60
WF_VAL_DAYS = 30       # sits between train and test, used only to fit shrinkage
WF_MIN_TRAIN_DAYS = 90  # below this a fold is skipped rather than trusted


def _wf_models():
    """Fresh unfitted models with FIXED hyperparameters (no per-fold tuning)."""
    dev = "cuda" if HAS_XGB_GPU else "cpu"
    xgb_p = dict(n_estimators=300, learning_rate=0.05, max_depth=6,
                 subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                 device=dev, n_jobs=1 if dev == "cuda" else -1,
                 random_state=RANDOM_STATE)
    lgb_p = dict(n_estimators=300, learning_rate=0.05, num_leaves=48,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                 n_jobs=-1, verbosity=-1, random_state=RANDOM_STATE)
    return {
        "Ridge": make_pipeline(StandardScaler(),
                               Ridge(alpha=1.0, random_state=RANDOM_STATE)),
        "XGBoost": xgb.XGBRegressor(**xgb_p),
        "XGBoost_L1": xgb.XGBRegressor(objective="reg:absoluteerror", **xgb_p),
        "LightGBM": lgb.LGBMRegressor(**lgb_p),
        "LightGBM_L1": lgb.LGBMRegressor(objective="regression_l1", **lgb_p),
    }


ts_all = df["timestamp"]
last_hour = ts_all.max()
wf_rows = []

for i in reversed(range(WF_FOLDS)):  # oldest fold first
    test_end = last_hour - pd.Timedelta(days=i * WF_TEST_DAYS)
    test_start = test_end - pd.Timedelta(days=WF_TEST_DAYS)
    val_start = test_start - pd.Timedelta(days=WF_VAL_DAYS)

    tr = df[ts_all < val_start]
    va = df[(ts_all >= val_start) & (ts_all < test_start)]
    te = df[(ts_all >= test_start) & (ts_all < test_end)]

    if len(te) == 0 or len(va) == 0 or len(tr) < WF_MIN_TRAIN_DAYS * 24:
        print(f"\nFOLD ending {test_end.date()}: SKIPPED "
              f"(train={len(tr)} rows, val={len(va)}, test={len(te)})")
        continue

    smog_share = float(te["is_smog_season"].mean())
    print(f"\n{'=' * 78}")
    print(f"FOLD  test {test_start.date()} -> {test_end.date()}  "
          f"| train={len(tr)} val={len(va)} test={len(te)} "
          f"| smog season = {smog_share:.0%} of test")
    print("=" * 78)

    for h in TARGET_HORIZONS:
        delta_col, abs_col = f"aqi_delta_{h}h", f"aqi_target_{h}h"
        te_anchor = te["aqi"].to_numpy(dtype=float)
        te_abs = te[abs_col].to_numpy(dtype=float)
        va_anchor = va["aqi"].to_numpy(dtype=float)
        va_abs = va[abs_col].to_numpy(dtype=float)

        base = persistence_baseline(te, abs_col)
        wf_rows.append({"fold_end": test_end.date(), "horizon": h,
                        "model": "Persistence", "smog_share": smog_share,
                        **{k: base[k] for k in ("RMSE", "MAE", "R2")}})
        print(f"  {h}h  {'Persistence':16} RMSE={base['RMSE']:7.2f} "
              f"MAE={base['MAE']:7.2f} R2={base['R2']:8.3f}")

        for name, model in _wf_models().items():
            model.fit(tr[feature_cols], tr[delta_col])
            if USE_DELTA_SHRINKAGE:
                lam, _ = fit_delta_shrinkage(va_anchor, va_abs,
                                             model.predict(va[feature_cols]))
            else:
                lam = 1.0
            pred = reconstruct_absolute(te_anchor, model.predict(te[feature_cols]), lam)
            m = evaluate(te_abs, pred, name)
            won = beats_persistence(m, base)
            wf_rows.append({"fold_end": test_end.date(), "horizon": h,
                            "model": name, "smog_share": smog_share,
                            "shrinkage": lam, "beats_persistence": won,
                            **{k: m[k] for k in ("RMSE", "MAE", "R2")}})
            print(f"  {h}h  {name:16} RMSE={m['RMSE']:7.2f} MAE={m['MAE']:7.2f} "
                  f"R2={m['R2']:8.3f} lam={lam:.2f}"
                  f"{'   BEATS PERSISTENCE' if won else ''}")

wf = pd.DataFrame(wf_rows)
wf.to_csv("reports/walk_forward.csv", index=False)

print("\n" + "=" * 78)
print("WALK-FORWARD SUMMARY — averaged over folds")
print("=" * 78)
for h in TARGET_HORIZONS:
    sub = wf[wf["horizon"] == h]
    pers_rmse = sub.loc[sub["model"] == "Persistence", "RMSE"].mean()
    pers_mae = sub.loc[sub["model"] == "Persistence", "MAE"].mean()
    agg = (sub[sub["model"] != "Persistence"]
           .groupby("model")
           .agg(mean_RMSE=("RMSE", "mean"), mean_MAE=("MAE", "mean"),
                mean_R2=("R2", "mean"), folds_won=("beats_persistence", "sum"),
                folds=("beats_persistence", "size"))
           .sort_values("mean_RMSE"))
    # Negative = better than persistence. This is the number that decides.
    agg["RMSE_vs_pers_%"] = (agg["mean_RMSE"] / pers_rmse - 1) * 100
    agg["MAE_vs_pers_%"] = (agg["mean_MAE"] / pers_mae - 1) * 100
    print(f"\n{h}h — persistence mean RMSE={pers_rmse:.2f} MAE={pers_mae:.2f}")
    print(agg.round(3).to_string())


## Done

Download from the Colab file browser if you want local copies:
- `reports/model_comparison.csv`
- `reports/model_comparison_{24,48,72}h.csv`
- `reports/walk_forward.csv` (per-fold metrics from section 9)
- `reports/shap_summary_{h}h.png` (if a tree model won)

Then tell your local agent: which model won each horizon, and whether each beat persistence.
